In [ ]:
import io, json, contextlib
import numpy as np, matplotlib.pyplot as plt
from matplotlib import colormaps
from scipy.integrate import solve_ivp

# Reuse the parent notebook up to and including scan_relic_line: dof tables,
# sigmavrel, the R ratio, the Kling data, dYdxcore and the scanner. Nothing
# past that point is touched, so the integral thermal averaging never loads.
_src = [''.join(c['source']) for c in json.load(open('Relic_Abundance.ipynb'))['cells']
        if c['cell_type'] == 'code']
_stop = next(i for i, s in enumerate(_src) if 'def scan_relic_line' in s) + 1
with contextlib.redirect_stdout(io.StringIO()):
    for _s in _src[:_stop]:
        if 'plt.show' in _s or '= scan_relic_line(' in _s:
            continue
        exec(compile(_s, '<parent>', 'exec'))

In [ ]:
# Every level, self-conjugate throughout: no 1/2 in the collision term, no
# compensating 2 in the relic normalisation. Defined here rather than reused
# so this cell is the only place the convention is set.

N, SVT = 1, 3e-9

def sig0_req(xf, mchi, n=N):
    T = mchi/xf
    return (3.79*(n+1)*xf**(n+1)*(2*np.pi**2/45)*gstarsT0*T0**3
            /((gstarsfun(T)/np.sqrt(gstarfun(T)))*mpl*3*Mpl**2*H0**2*OmegaDM))

def xf_kt(vare, ma, al, aD, mchi, xf=20.0, n=N):
    A = np.log((n+1)*0.038*gchi*mpl*mchi*sigmavrel(vare, ma, al, aD, mchi, 1)
               /np.sqrt(gstarfun(mchi/xf)))
    return A - (n+1/2)*np.log(A)

def sv_target(xf, mchi, n=N):
    T = mchi/xf
    return ((n+1)*np.pi/(9*np.sqrt(10))*np.sqrt(gstarfun(T))/gstarsfun(T)
            *gstarsT0*T0**3/(H0**2*Mpl**3*OmegaDM)*xf)

def xf_gamma(mchi, n=N, tol=1e-12, itmax=300):
    mchi = np.asarray(mchi, float); x = 20.0*np.ones_like(mchi)
    for _ in range(itmax):
        xn = np.log(np.sqrt(90/np.pi**2)*gchi*Mpl/(2*np.pi)**1.5*mchi
                    *sv_target(x, mchi, n)*np.sqrt(x)/np.sqrt(gstarfun(mchi/x)))
        if np.max(np.abs(xn - x)) < tol:
            return xn
        x = xn
    return x

def L1(ma, al, aD, m):
    return np.sqrt(SVT/sigmavrel(1, ma, al, aD, m, 20))

def L2(ma, al, aD, m):
    return np.sqrt(SVT/sigmavrel(1, ma, al, aD, m, np.log(SVT*mpl*m)))

def L3(ma, al, aD, m):
    return np.sqrt(sig0_req(20, m)/sigmavrel(1, ma, al, aD, m, 1))

def L4(ma, al, aD, m):
    m = np.asarray(m, float)
    su = sigmavrel(1, ma, al, aD, m, 1)
    xf = 20.0*np.ones_like(m); eps = np.sqrt(sig0_req(xf, m)/su)
    for _ in range(200):
        xf = xf_kt(eps, ma, al, aD, m, xf)
        new = np.sqrt(sig0_req(xf, m)/su)
        if np.max(np.abs(np.log(new/eps))) < 1e-12:
            return new
        eps = new
    return eps

def L5(ma, al, aD, m):
    xf = xf_gamma(m)
    return np.sqrt(sv_target(xf, m)/sigmavrel(1, ma, al, aD, m, xf))

def omega_sc(ma, vare, alphaD, mchi_over_ma=0.6):
    mchi = mchi_over_ma*ma
    rhs = lambda x, Y: 2*dYdxcore(x, Y, 2, mchi,
                                  sigmavrel(vare, ma, alpha, alphaD, mchi, x))
    sol = solve_ivp(rhs, (1e1, 1e3), [Yeq(1e1, 2, mchi)], method="Radau",
                    rtol=1e-6, atol=1e-25)
    return omega(mchi, sol.y[0][-1]) if sol.success else np.nan

In [ ]:
ALPHAD, RATIO = 0.1, 0.6

ma_c = np.unique(np.concatenate([
    np.logspace(np.log10(0.012), np.log10(0.40), 13),
    np.logspace(np.log10(0.40),  np.log10(1.20), 22),
    np.logspace(np.log10(1.20),  np.log10(1.62),  3)]))

LEVELS = [(r"1: fixed $\langle\sigma v\rangle$, $x_f=20$",          L1),
          (r"2: fixed $\langle\sigma v\rangle$, $x_f$ from a log",  L2),
          (r"3: relic $\sigma_0$, $x_f=20$",                        L3),
          (r"4: relic $\sigma_0$, $x_f$ from KT",                   L4),
          (r"5: $\langle\sigma v\rangle_f$, $x_f$ from $\Gamma=H$",  L5)]

eps_lv = [np.array([float(f(np.array(m), alpha, ALPHAD, np.array(RATIO*m)))
                    for m in ma_c]) for _, f in LEVELS]
eps_bz, _ = scan_relic_line(ma_c, 1e-6, 5e-3, alphaD=ALPHAD,
                            mchi_over_ma=RATIO, omega_fun=omega_sc, verbose=False)

curves = [(lab, e) for (lab, _), e in zip(LEVELS, eps_lv)]
curves.append(("6: Boltzmann", eps_bz))
eps_kl = np.exp(np.interp(np.log(ma_c), np.log(ma_kling), np.log(eps_kling)))

# The five closed forms are an ordered ladder, so they take one perceptually
# uniform ramp; five steps clear the normal-vision separation floor (min
# dE 19), six do not, which is why the Boltzmann line is not part of it. It
# and Kling are the two references and get their own marks. A linestyle rides
# along as secondary encoding so identity is never colour alone.
RAMP   = [colormaps["viridis"](t) for t in np.linspace(0.0, 1.0, 5)]
COLORS = RAMP + ["#111111"]
STYLES = [(0, (1, 2)), (0, (4, 2)), (0, (7, 2)), (0, (9, 2, 1, 2)), (0, (12, 3)), "solid"]
WIDTHS = [2.0]*5 + [2.8]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

ax.set_xlabel(r"$m_{A'}$(GeV)")
ax.set_ylabel(r"$\varepsilon$")
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, which='major', linestyle='-', linewidth=0.75, alpha=0.75)
ax.minorticks_on()
ax.grid(True, which='minor', linestyle='-', linewidth=0.5, alpha=0.75)
ax.set_axisbelow(True)

ax.plot(ma_kling, eps_kling, color='#d62728', linewidth=3.5, label="Kling", zorder=1)
for (lab, e), col, ls, lw in zip(curves, COLORS, STYLES, WIDTHS):
    ax.plot(ma_c, e, color=col, linewidth=lw, linestyle=ls, label=lab, zorder=2)

ax.text(0.03, 0.97,
        "\n".join([r"$\alpha_D = %g$" % ALPHAD,
                   r"$m_\chi = %g\, m_{A'}$" % RATIO,
                   r"$\Omega_{DM} = %g$" % OmegaDM,
                   r"self-conjugate"]),
        transform=ax.transAxes, va='top', ha='left', fontsize=18,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor='#7e7e7e', alpha=0.85))

ax.legend(loc='lower right', frameon=False, fontsize=15)
ax.set_xbound(1e-2, 1.7)
ax.set_ybound(1e-5, 3e-3)
plt.savefig("figures/convergence_lines.pdf", bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)

for axi, (lab, e), col in zip(axes.ravel(), curves, COLORS):
    r = e/eps_kl
    axi.axhspan(0.9, 1.1, color='#7e7e7e', alpha=0.18, zorder=0)
    axi.axhline(1.0, color='#d62728', linewidth=2.0, zorder=1)
    axi.plot(ma_c, r, color=col, linewidth=2.5, zorder=2)
    axi.set_xscale('log')
    axi.grid(True, which='major', linestyle='-', linewidth=0.6, alpha=0.6)
    axi.set_axisbelow(True)
    axi.set_title(lab, fontsize=15)
    axi.text(0.04, 0.06, r"median $%.3f$" % np.median(r[np.isfinite(r)]),
             transform=axi.transAxes, fontsize=15,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                       edgecolor='#7e7e7e', alpha=0.85))

axes[0, 0].set_ybound(0.4, 1.6)
for axi in axes[1]:
    axi.set_xlabel(r"$m_{A'}$(GeV)")
for axi in axes[:, 0]:
    axi.set_ylabel(r"$\varepsilon\,/\,\varepsilon_{\rm Kling}$")

plt.tight_layout()
plt.savefig("figures/convergence_ratio.pdf", bbox_inches='tight')
plt.show()